# Generate Ensembl-ID Mapping

In [1]:
import anndata as ad
import scanpy as sc
from sklearn.linear_model import LogisticRegression
from skopt import BayesSearchCV
from skopt.space import Real, Categorical
from custom_stopper import CustomStopper
# For saving results on HPC Cluster
import joblib
import pandas as pd
import os

# Load training data
#adata = ad.io.read_h5ad('../data/CellTypistDataset/global_annotated.h5ad')
adata = ad.io.read_h5ad('../data/CellTypistDataset/CountAdded_PIP_global_object_for_cellxgene_annotated.h5ad')

# Filter blood data
adata = adata[adata.obs['Organ'] == 'BLD'].copy()
print(adata)

# Use raw data instead of already preprocessed data
adata.X = adata.layers['counts'].copy()

# Preprocessing

# mitochondrial genes, "MT-" for human, "Mt-" for mouse
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)

# Remove mitochondrial, ribosomal and hemoglobin
adata = adata[:, ~adata.var["mt"]].copy()
adata = adata[:, ~adata.var["ribo"]].copy()
adata = adata[:, ~adata.var["hb"]].copy()

# Doublet Detection
sc.pp.scrublet(adata, batch_key="Donor")
adata = adata[adata.obs['predicted_doublet'] == False].copy()


# Normalization

# Saving count data
adata.layers["counts"] = adata.X.copy()

# Normalizing to median total counts
sc.pp.normalize_total(adata, target_sum=1e4)
# Logarithmize the data
sc.pp.log1p(adata)

# Filtering Highly variable genes
print('Before filtering highly variable genes ---')
print(adata)

sc.pp.highly_variable_genes(adata, n_top_genes=10000)

# Apply filter
adata = adata[:, adata.var['highly_variable']].copy()

print('After filtering highly variable genes ---')
print(adata)

# Create train test split

# All Donors: ['621B', '637C', 'A35', 'A36', 'D496', 'D503']
donor_train = ['637C', 'A35', 'A36', 'D503']
donor_test = ['621B', 'D496']

adata_train = adata[
    adata.obs["Donor"].isin(donor_train)
].copy()

adata_test = adata[
    adata.obs["Donor"].isin(donor_test)
].copy()

# Check split
print(adata_train.obs['Donor'].unique())
print(adata_test.obs['Donor'].unique())

# Prepare Data for training
X_train = adata_train.to_df()
gene_names_train = adata_train.var_names
y_train = adata_train.obs['scumi-annotation']

X_test = adata_test.to_df()
gene_names_test = adata_test.var_names
y_test = adata_test.obs['scumi-annotation']

AnnData object with n_obs × n_vars = 27620 × 36473
    obs: 'Organ', 'Donor', 'Chemistry', 'Cell_category', 'Predicted_labels_CellTypist', 'Majority_voting_CellTypist', 'Majority_voting_CellTypist_high', 'Manually_curated_celltype', 'Sex', 'Age_range', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'doublet_score', 'predicted_doublet', 'scumi-annotation'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'Age_range_colors', 'Sex_colors', 'scrublet'
    obsm: 'X_umap'
    layers: 'counts'
Before filtering highly variable genes 

In [2]:
# get_gene_mapping.py
import json
import pandas as pd
from gprofiler import GProfiler

# 1. Lädst du hier kurz deine Daten (nur die Spalten/Gene reichen, um Zeit zu sparen)
# Passe den Pfad zu deinem originalen train_data an!
gene_symbols = X_train.columns.tolist()

# 2. g:Profiler abfragen (auf dem Login-Knoten mit Internet)
print(f"Frage g:Profiler für {len(gene_symbols)} Gene an...")
gp = GProfiler(return_dataframe=True)
df_map = gp.convert(
    organism='hsapiens',
    query=gene_symbols,
    target_namespace='ENSG'
)

# 3. Dictionary säubern
df_map_clean = df_map[df_map['converted'].notna() & (df_map['converted'] != 'None')]
symbol_to_ensembl = dict(zip(df_map_clean['incoming'], df_map_clean['converted']))

# 4. Als JSON-Datei auf dem Cluster abspeichern
mapping_file = "gene_symbol_to_ensembl.json"
with open(mapping_file, "w") as f:
    json.dump(symbol_to_ensembl, f, indent=4)

print(f"Anzahl Mappings: {len(symbol_to_ensembl.keys())}")
print(f"Erfolgreich! Mapping wurde in '{mapping_file}' gespeichert.")

Frage g:Profiler für 10000 Gene an...
Anzahl Mappings: 7366
Erfolgreich! Mapping wurde in 'gene_symbol_to_ensembl.json' gespeichert.


In [7]:
# Welche Gene wurden NICHT gefunden?
missing_genes = list(set(gene_symbols) - set(symbol_to_ensembl.keys()))

print(f"Anzahl nicht gefundener Gene: {len(missing_genes)}")
print("Beispiele für nicht gefundene Gene:")
print(missing_genes[:50])  # Zeigt die ersten 50 Blindgänger an

Anzahl nicht gefundener Gene: 2634
Beispiele für nicht gefundene Gene:
['AL162377.1', 'AL391840.3', 'TTC26', 'AP001189.1', 'AC003035.1', 'AC002072.1', 'AC108479.3', 'AC072026.2', 'AC009630.3', 'AC079142.1', 'AC120114.1', 'AL022323.4', 'AP003472.2', 'AP000553.2', 'AC010463.3', 'AP001437.2', 'AC079804.3', 'AC090241.2', 'AC138028.2', 'AL359237.1', 'AC120049.1', 'AL360012.1', 'AC026474.1', 'AP000253.1', 'H2AFY2', 'AL356441.1', 'AC091053.1', 'AL136018.1', 'AC007378.1', 'AC084117.1', 'AC004877.1', 'AF131215.6', 'AC096564.2', 'AC022182.1', 'AC020558.1', 'AC105411.1', 'AC092849.1', 'AC012291.3', 'AC010931.2', 'AL391001.1', 'AL121929.3', 'AC008569.1', 'CXorf21', 'AC005050.3', 'AL445213.2', 'Z97192.3', 'AL021707.9', 'AC022903.2', 'AC083843.2', 'AL359962.3']


In [8]:
import mygene
import json

# ... Dein bisheriger g:Profiler Code ...
# (Ergebnis ist symbol_to_ensembl)

missing_genes = list(set(gene_symbols) - set(symbol_to_ensembl.keys()))

if missing_genes:
    print(f"Versuche {len(missing_genes)} verbleibende Gene mit MyGene zu retten...")
    mg = mygene.MyGeneInfo()
    
    # querymany durchsucht auch Aliases und alte Symbole
    res = mg.querymany(missing_genes, scopes='symbol,alias', fields='ensembl.gene', species='human', as_dataframe=True)
    
    # Ergebnisse extrahieren
    if 'ensembl.gene' in res.columns:
        # MyGene gibt manchmal Listen aus, falls ein Gen mehrere IDs hat -> wir nehmen das erste Element oder droppen NaNs
        rescued_df = res['ensembl.gene'].dropna()
        
        rescued_dict = {}
        for idx, val in rescued_df.items():
            # Falls MyGene eine Liste von IDs zurückgibt, nimm die erste
            if isinstance(val, list):
                rescued_dict[idx] = val[0]['gene']
            elif isinstance(val, dict):
                rescued_dict[idx] = val['gene']
            else:
                rescued_dict[idx] = str(val)
                
        # Das g:Profiler Dictionary mit den geretteten Genen updaten
        symbol_to_ensembl.update(rescued_dict)

print(f"Finale Anzahl gemappter Gene nach Rettungsaktion: {len(symbol_to_ensembl)}")

# Jetzt erst als JSON speichern
#with open("gene_symbol_to_ensembl.json", "w") as f:
#    json.dump(symbol_to_ensembl, f, indent=4)

Versuche 2634 verbleibende Gene mit MyGene zu retten...


12 input query terms found dup hits:	[('HIST1H2BG', 2), ('C20orf197', 2), ('LINC00545', 2), ('HIST1H2BC', 2), ('LINC02634', 2), ('PRR34-A
2397 input query terms found no hit:	['AL162377.1', 'AL391840.3', 'AP001189.1', 'AC003035.1', 'AC002072.1', 'AC108479.3', 'AC072026.2', '


Finale Anzahl gemappter Gene nach Rettungsaktion: 7564
